## 06 - Lines and Distances

### Goal

- Last lesson we loaded the WDO package and used some of the given spatial functions. Now we are going to put package to use and calculate many distances and draw some lines. 
- Using any file that has "world cities" in the name:
  
  - [WorldCitiesGeo](../data/WorldCitiesGeo/) (and all its geojson files in the directory)
  - [world_cities_fixed.json](../data/world_cities_fixed.json)
  - [world_cities_by_time-zone.json](../data/world_cities_by_time-zone.json)
  
- These contain ALL THE CITIES IN THE WORLD! Well ... most of the cities.  You know like the ones that ... have people.  I mean `Burkburnett` probably isn't in the file, but it does have people, so ... (update ... I just checked, and Burk is in the file!)
  
- Keep reading


### Tasks

- Read in a file containing the locations to cities all over the world ([world_cities_large.json](./../data/world_cities_large.json)).
- We would like to draw a line from MSU (or close to it) to each city in the file, but thats not feasible as the file is too large. 
- Looking at the options (files) above, you should have plenty of choices for filtering down to a manageable size of cities to draw lines to. 


## Possible Bonus

- Randomly choose cities from the file, and stop choosing when the distance goes over some threshold (e.g 10000km).

In [4]:
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

EARTH_RADIUS_KM = 6371.0

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    phi1 = radians(lat1)
    phi2 = radians(lat2)
    dphi = radians(lat2 - lat1)
    dlmb = radians(lon2 - lon1)
    a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlmb / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return EARTH_RADIUS_KM * c

debug = False
cwd = Path.cwd()

target = cwd / "data" / "countries.geojson"
exists = target.exists()

if debug:
    print("CWD:", cwd)
    print("Target:", target)
    print("Exists:", exists)

assert exists, (
    f"\n❌ ERROR: File not found:\n{target}\n"
    "Check spelling and folder structure."
)

**FYI:**

These previous three lessons will be helpful: 
- [03-Style_W_Logic](03-Style_W_Logic.ipynb)
- [04-Distance](04-Distance.ipynb)
- [05-Geo_and_Json_overview](05-Geo_and_Json_overview.ipynb)



In [5]:
import json
import random
import folium

MSU_LAT =  33.87372
MSU_LON = -98.51947

THRESHOLD_KM = 10_000
MAX_CITIES   = 120

cities_path = cwd / "data" / "world_cities_fixed.json"
with open(cities_path, encoding="utf-8") as f:
    all_cities = json.load(f)

# Compute distance from MSU for every city, keep only those within threshold
in_range = []
for city in all_cities:
    dist = haversine_km(MSU_LAT, MSU_LON, float(city["lat"]), float(city["lon"]))
    if dist <= THRESHOLD_KM:
        in_range.append({**city, "dist_km": dist})

print(f"{len(in_range):,} of {len(all_cities):,} cities are within {THRESHOLD_KM:,} km of MSU")

# Randomly sample a map-friendly subset from the cities that passed the threshold
random.seed(42)
selected = random.sample(in_range, min(MAX_CITIES, len(in_range)))
selected.sort(key=lambda c: c["dist_km"])

print(f"Plotting {len(selected)} randomly sampled cities")
print(f"Distance range: {selected[0]['dist_km']:,.0f} km  →  {selected[-1]['dist_km']:,.0f} km")

m = folium.Map(location=[MSU_LAT, MSU_LON], zoom_start=2)

folium.Marker(
    location=[MSU_LAT, MSU_LON],
    popup="MSU — Wichita Falls, TX",
    icon=folium.Icon(color="blue", icon="star"),
).add_to(m)

for city in selected:
    lat  = float(city["lat"])
    lon  = float(city["lon"])
    dist = city["dist_km"]
    name = city["city-name"]

    if dist < 3_000:
        color = "#27AE60"
    elif dist < 7_000:
        color = "#F39C12"
    else:
        color = "#E74C3C"

    folium.PolyLine(
        locations=[[MSU_LAT, MSU_LON], [lat, lon]],
        color=color,
        weight=1,
        opacity=0.55,
        tooltip=f"{name} — {dist:,.0f} km",
    ).add_to(m)

    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.85,
        tooltip=f"{name} — {dist:,.0f} km",
    ).add_to(m)

m

100,450 of 149,102 cities are within 10,000 km of MSU
Plotting 120 randomly sampled cities
Distance range: 92 km  →  9,905 km


This lesson helps with **📏 Milestone 2** 